# Football-CV: Phase 0 & Phase 1 Pipeline
### End-to-End Kaggle Runner for Dataset Acquisition, Calibration, and YOLOv8 Training

This notebook runs the complete pipeline on Kaggle GPU:
1. **Environment Setup & Dependencies** (`ultralytics`, `roboflow`, `huggingface_hub`)
2. **Phase 0 Data Acquisition**: Download SoccerNet-V3 (HF) & License-Compliant Roboflow datasets
3. **4-Class Remapping & Manifest Generation** (`player`, `goalkeeper`, `referee`, `ball`)
4. **Strict Match-Level Splitting** (Zero video frame leakage across train/val/test)
5. **Dataset Distribution & Small-Object (Ball) Audit**
6. **Pitch Homography Calibration** (Pixels $\to$ 105x68m pitch meters)
7. **Phase 1 YOLOv8 Training** (High-res `imgsz=960`, small-object preservation)
8. **Per-Class Metrics & ONNX Export**

## 1. Environment Setup & Clone Repository

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow huggingface_hub pyyaml tqdm pandas opencv-python-headless

# Clone your repository (or run directly if uploaded as a Kaggle dataset)
# !git clone https://github.com/<YOUR_USERNAME>/soccer_monitor.git
# %cd soccer_monitor

import os
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Configure API Keys (Kaggle Secrets or Environment Variables)

In [ ]:
# Set your Roboflow API key (can also use Kaggle Secrets: from kaggle_secrets import UserSecretsClient)
os.environ["ROBOFLOW_API_KEY"] = ""  # Fill in if downloading from Roboflow Universe

## 3. Phase 0 — Data Acquisition (SoccerNet-V3 & Roboflow)

In [ ]:
# Download SoccerNet-V3 GameState subset (MIT license, no NDA required)
!python scripts/download_soccernet.py --max-matches 5

# Download license-compliant Roboflow dataset (rejects any NC datasets automatically)
# If you have a specific approved Roboflow project:
# !python scripts/download_roboflow.py --workspace roboflow-jvuqo --project football-players-detection-3zvbc --version 1

## 4. Phase 0 — Dataset Staging, Class Remapping & Manifest

In [ ]:
# Remap all raw datasets into unified 4 classes:
# 0: player, 1: goalkeeper, 2: referee, 3: ball
# Generates data/manifest.csv with full provenance
!python scripts/merge_datasets.py

## 5. Phase 0 — Strict Match-Level Split (Zero Leakage)

In [ ]:
# Partition at the MATCH level: every frame from a match is in ONE split only
!python scripts/split_by_match.py --train-ratio 0.70 --val-ratio 0.15 --test-ratio 0.15

## 6. Phase 0 — Dataset Validation & Ball Ratio Audit

In [ ]:
# Check per-class distributions and verify ball representation is >= 10% of players
!python scripts/validate_dataset.py --data-dir data/labeled

## 7. Phase 0 — Pitch Homography Calibration (Parallel Task)

In [ ]:
# Generates template landmark mappings and computes homography matrix
!python scripts/calibrate_pitch.py --venue mobolaji_johnson_arena --cam cam_main

# Display calibrated homography JSON
import json
calib_file = "data/calibration/template_landmarks.json"
if os.path.exists(calib_file):
!python scripts/calibrate_pitch.py --venue mobolaji_johnson_arena --cam cam_main --points-file data/calibration/template_landmarks.json

## 8. Phase 1 — Detection Model Training (YOLOv8)
We train at higher resolution (`imgsz=960`) to preserve small ball features.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 small
model = YOLO('yolov8s.pt')

# Fine-tune on the merged 4-class football dataset
results = model.train(
    data='configs/dataset.yaml',
    epochs=50,
    imgsz=960,
    batch=8,
    optimizer='AdamW',
    lr0=0.001,
    mosaic=0.8,
    fliplr=0.5,
    flipud=0.0,
    name='football_cv_phase1',
    project='runs/train'
)

## 9. Phase 1 — Validation & Per-Class Metric Inspection
Verify Class 3 (`ball`) AP50 to ensure small-object convergence.

In [ ]:
# Validate the best checkpoint
metrics = model.val()

print("\n--- Per-Class Detection AP50 ---")
class_names = ['player', 'goalkeeper', 'referee', 'ball']
for idx, name in enumerate(class_names):
    if idx < len(metrics.box.ap50):
        print(f"[{idx}] {name:<12}: AP50 = {metrics.box.ap50[idx]:.4f}")
print(f"Aggregate mAP50    : {metrics.box.map50:.4f}")
print(f"Aggregate mAP50-95 : {metrics.box.map:.4f}")

## 10. Phase 1 — Export to ONNX for High-Speed Inference

In [ ]:
# Export trained weights to ONNX format
success = model.export(format='onnx', imgsz=960, dynamic=True)
print(f"Exported ONNX model path: {success}")